In [ ]:
# ============================================================
# MODELO DE SCORING DE RIESGO CREDITICIO (MÓDULO B) - COMPLETO
# ============================================================

# ---------- INSTALACIÓN (correr 1 sola vez por sesión) ----------
!pip install xgboost gdown -q

# ---------- IMPORTAR LIBRERÍAS ----------
import pandas as pd
import numpy as np
import requests
import glob
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

# ============================================================
# PASO 1: DESCARGAR EL DATASET DESDE DRIVE (carpeta compartida)
# ============================================================

!gdown --folder "https://drive.google.com/drive/folders/1H0DKmzeEWhMzRHmuGgDoxDzSjuJSGCWs"

# Buscar el CSV automáticamente en todo /content (dentro o fuera de carpetas)
archivos_csv = glob.glob('/content/**/application_train.csv', recursive=True)
print("Archivos encontrados:", archivos_csv)

ruta_csv = archivos_csv[0] if archivos_csv else '/content/application_train.csv'
print("Cargando desde:", ruta_csv)

# ============================================================
# PASO 2: CARGA DE DATOS Y ANÁLISIS EXPLORATORIO (EDA)
# ============================================================

df = pd.read_csv(ruta_csv)

print("--- INFORMACIÓN DEL DATASET ---")
print(f"Cantidad de solicitudes evaluadas: {df.shape[0]}")
print(f"Cantidad de variables por cliente: {df.shape[1]}\n")

# 2. Seleccionar las variables financieras clave que pide el banco
columnas = ['SK_ID_CURR', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'DAYS_BIRTH', 'TARGET']
df = df[columnas]

print("--- RESUMEN ESTADÍSTICO ---")
print(df.describe().T)

# 3. Transformar la edad: DAYS_BIRTHDAY viene en días negativos
df['AGE'] = (df['DAYS_BIRTH'] / -365).astype(int)

# 4. Diagnosticar el riesgo crediticio del portafolio
conteo = df['TARGET'].value_counts()
print("--- DIAGNÓSTICO DE RIESGO CREDITICIO ---")
print(f"Clientes que pagaron a tiempo (0): {conteo.get(0, 0)}")
print(f"Clientes morosos (1): {conteo.get(1, 0)}")
print(f"Tasa de morosidad del portafolio: {(df['TARGET'].mean() * 100):.2f}%\n")

# 5. Tratamiento de valores nulos (rellenar con la mediana)
print("--- VALORES NULOS POR COLUMNA ---")
print(df.isnull().sum())

df = df.fillna(df.median(numeric_only=True))
print("\nNulos después del relleno:")
print(df.isnull().sum().sum())

# ============================================================
# PASO 3: ENTRENAR EL MODELO (SCORECARD DE RIESGO)
# ============================================================

# Variables de entrada (X) y objetivo (y)
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

# Separar 80% entrenamiento / 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entrenar XGBoost (compensa el desbalance de clases)
peso = (y_train == 0).sum() / (y_train == 1).sum()
model = XGBClassifier(scale_pos_weight=peso, random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

# Predecir sobre datos que el modelo NUNCA vio
predicciones = model.predict(X_test)
prob_morosidad = model.predict_proba(X_test)[:, 1]  # PD = Probabilidad de Incumplimiento

# Evaluar el modelo
print("--- MATRIZ DE CONFUSIÓN ---")
print("(Filas = real, Columnas = predicho. [[correctos 0, falsos morosos], [falsos 0, correctos 1]])")
print(confusion_matrix(y_test, predicciones))

print("\n--- REPORTE DE CLASIFICACIÓN ---")
print(classification_report(y_test, predicciones))

print(f"\nAUC-ROC: {roc_auc_score(y_test, prob_morosidad):.4f}")
print("AUC-ROC mide la capacidad de separar morosos de no morosos.")
print("0.5 = aleatorio, 1.0 = perfecto. Cuanto más cerca de 1, mejor.")

# Tabla de resultados para el banco
resultados = pd.DataFrame({
    'SK_ID_CURR': df['SK_ID_CURR'].iloc[X_test.index].values,
    'TASA_MOROSIDAD_PCT': np.round(prob_morosidad * 100, 2)
})
print("\n--- RESULTADOS PARA EL BANCO (MUESTRA) ---")
print(resultados.head(10))

Retrieving folder contents
Processing file 1eGP3Qenj6hDIl0McGFoHxSN0L1cMPXip application_train.csv
Processing file 1jxPaXOY_IO5txG70KIfMYnqF2J3tlarFLso-cN4JvAM Documento Ejecutivo de Propuesta Comercial (RFP / FinOps)
Processing file 1sZfh1L7IDB0tcP3QOmWueJ59u1m55P_3SF86n6lm77k GUIA HECTOR
Processing file 1pyX-ck3RUqVyk2KUfZZSn9Cz65QVUJ5U Sistema de Alerta Temprana de Riesgo Financiero y Automatización ERP
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1eGP3Qenj6hDIl0McGFoHxSN0L1cMPXip
From (redirected): https://drive.google.com/uc?id=1eGP3Qenj6hDIl0McGFoHxSN0L1cMPXip&confirm=t&uuid=f4207984-e755-4ed7-872f-9ce943d9f37f
To: /content/Sistema de Alerta Temprana de Riesgo Financiero y Automatización ERP/application_train.csv
100% 166M/166M [00:01<00:00, 91.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1jxPaXOY_IO5txG70KIfMYnqF2J3tlarFLso-cN4JvAM
F

In [4]:
# ============================================================
# PASO 4: ENVIAR RESULTADOS AL BACKEND DE JAVA (NGROK)
# ============================================================
# URL pública de Ngrok que expone la API de tu compañero
URL_NGROK = "https://shucking-rename-scrambler.ngrok-free.dev"

# Ngrok Free muestra una pantalla de advertencia en el navegador;
# este header la evita al hacer peticiones desde código.
HEADERS = {"ngrok-skip-browser-warning": "true"}

print("--- ENVÍO DE RESULTADOS AL BANCO (PRIMEROS 5 CLIENTES) ---")
for fila in resultados.head(5).itertuples():
    url = f"{URL_NGROK}/api/riesgos/actualizar?clienteId={fila.SK_ID_CURR}&tasa={fila.TASA_MOROSIDAD_PCT}"
    try:
        response = requests.put(url, timeout=15, headers=HEADERS)
        print(f"Cliente {fila.SK_ID_CURR} | Tasa {fila.TASA_MOROSIDAD_PCT}% | {response.status_code} -> {response.text}")
    except Exception as e:
        print(f"Cliente {fila.SK_ID_CURR}: ERROR {e}")

--- ENVÍO DE RESULTADOS AL BANCO (PRIMEROS 5 CLIENTES) ---
Cliente 396899 | Tasa 46.59000015258789% | 200 -> Cliente no encontrado.
Cliente 322041 | Tasa 55.75% | 200 -> Cliente no encontrado.
Cliente 220127 | Tasa 34.5099983215332% | 200 -> Cliente no encontrado.
Cliente 251531 | Tasa 45.86000061035156% | 200 -> Cliente no encontrado.
Cliente 345558 | Tasa 42.56999969482422% | 200 -> Cliente no encontrado.


In [5]:
import requests

# Forzamos la evaluación de tu cliente cargado por Power Automate
cliente_id = 100002
tasa_calculada = 7.45  # Probabilidad estimada por su IA

url_puente = "https://shucking-rename-scrambler.ngrok-free.dev/"
url_final = f"{url_puente}/api/riesgos/actualizar?clienteId={cliente_id}&tasa={tasa_calculada}"

response = requests.put(url_final)
print(f"Resultado final del banco: {response.text}")

Resultado final del banco: Riesgo bancario actualizado con éxito por la celda de IA.
